# Imports

In [1]:
import json
import os
from tqdm import tqdm
from collections import defaultdict, Counter
import matplotlib.pyplot as plt
import numpy as np


# Load in book works and see structure

In [2]:
books = os.path.join("./Datasets", "goodreads_books.json") # descriptions here
books_meta = []
with open(books, 'r') as books_data:
    count = 0
    for line in books_data:
        books_meta.append(json.loads(line))
        count += 1
        if count >= 10000: #TODO: note stopping early here
            break

print(books_meta[3]['description'])
print(books_meta[3])

#Note this has book id and ibsn (can serve as mapping)

#HAS TEH DESCRIPTIOSN

Addie Downs and Valerie Adler were eight when they first met and decided to be best friends forever. But, in the wake of tragedy and betrayal during their teenage years, everything changed. Val went on to fame and fortune. Addie stayed behind in their small Midwestern town. Destiny, however, had more in store for these two. And when, twenty-five years later, Val shows up at Addie's front door with blood on her coat and terror on her face, it is the beginning of a wild adventure for two women joined by love and history who find strength together that they could not find alone.
{'isbn': '0743294297', 'text_reviews_count': '3282', 'series': [], 'country_code': 'US', 'language_code': 'eng', 'popular_shelves': [{'count': '7615', 'name': 'to-read'}, {'count': '728', 'name': 'chick-lit'}, {'count': '673', 'name': 'currently-reading'}, {'count': '404', 'name': 'fiction'}, {'count': '152', 'name': 'books-i-own'}, {'count': '119', 'name': 'jennifer-weiner'}, {'count': '82', 'name': 'chicklit'}, 

# Load in reviews and see structure

In [3]:
# reviews here

reviews_path = os.path.join("./Datasets", "goodreads_reviews_dedup.json") # descriptions here
book_reviews = []
with open(reviews_path, 'r') as reviews_data:
    count = 0
    for line in reviews_data:
        book_reviews.append(json.loads(line))
        count += 1
        if count >= 10000: #TODO: note stopping early here
            break

print(book_reviews[1])

{'user_id': '8842281e1d1347389f2ab93d60773d4d', 'book_id': '18245960', 'review_id': 'dfdbb7b0eb5a7e4c26d59a937e2e5feb', 'rating': 5, 'review_text': 'This is a special book. It started slow for about the first third, then in the middle third it started to get interesting, then the last third blew my mind. This is what I love about good science fiction - it pushes your thinking about where things can go. \n It is a 2015 Hugo winner, and translated from its original Chinese, which made it interesting in just a different way from most things I\'ve read. For instance the intermixing of Chinese revolutionary history - how they kept accusing people of being "reactionaries", etc. \n It is a book about science, and aliens. The science described in the book is impressive - its a book grounded in physics and pretty accurate as far as I could tell. Though when it got to folding protons into 8 dimensions I think he was just making stuff up - interesting to think about though. \n But what would happ

# find pct null (descriptions) values and number of reviews for each book -> find average number of positive anchors

In [4]:
# find size of reviews
with open(reviews_path, 'r') as reviews_data:
    review_count = 0
    for line in reviews_data:
        review_count += 1

# find size of books
with open(books, 'r') as books_data:
    book_count = 0
    for line in books_data:
        book_count += 1

print(review_count, book_count)
# num reviews = 9324443, book count = 2360655

9324443 2360655


In [5]:
# find books with null descriptions
    # save book_id of null and non null
null_desc_ct = 0
good_desc_ct = 0
good_book_ids = set()
with open(books, 'r') as books_data:
    for line in tqdm(books_data, total=book_count):
        try:
            line = json.loads(line)
            if not line['description']:
                null_desc_ct += 1
            else:
                good_desc_ct += 1
                good_book_ids.add(line['book_id'])
        except json.JSONDecodeError:
            null_desc_ct += 1



100%|██████████| 2360655/2360655 [01:11<00:00, 32900.95it/s]


In [6]:

# find books with no reviews
    # save book_id
null_rev_ct = 0
good_rev_ct = 0
matched_revs = 0
num_per_book_id = defaultdict(int)

with open(reviews_path, 'r') as reviews_data:
    for line in tqdm(reviews_data, total=review_count):
        try:
            line = json.loads(line)
            if not line['review_text']:
                null_rev_ct += 1
            else:
                good_rev_ct += 1
                b_id = line['book_id']
                if b_id in good_book_ids:
                    matched_revs += 1
                    num_per_book_id[b_id] += 1

        except json.JSONDecodeError:
            null_rev_ct += 1

print(f"null desc: {null_desc_ct}\ngood desc: {good_desc_ct}\nnull rev: {null_rev_ct}\ngood rev: {good_rev_ct}\nmatched revs: {matched_revs}")

100%|██████████| 9324443/9324443 [01:56<00:00, 80283.01it/s] 

null desc: 412233
good desc: 1948422
null rev: 4110
good rev: 9320333
matched revs: 8838351


In [7]:
# derived stats
# pct null desc
print(f"pct null desc: {null_desc_ct / (good_desc_ct + null_desc_ct)}")

# pct null rev
print(f"pct null rev: {null_rev_ct / (good_rev_ct + null_rev_ct)}")

# num matched rev per book
print(f"num matched per book: {matched_revs / good_desc_ct}")

# with 4.53 revs per book we are looking at groups of size 5.5 (so 4.5 possible + anchors)

pct null desc: 0.17462653373745846
pct null rev: 0.00044077699869043117
num matched per book: 4.536158491332986


In [8]:
# get dist of num reviews per

# vals in defaultdict num_per_book_id
freq_counts = Counter(num_per_book_id.values()) # may need to convert to counter
sorted_vals = sorted(freq_counts.keys())
freqs = [(freq_counts[val], val) for val in sorted_vals]

print(freqs)
# 690k w 1, 213k w 2, 107k w 3, 66k w 4, 45k w 5, 32k w 6, etc

# median calc 
median = np.median(list(num_per_book_id.values()))
print(f"median: {median}")

#TODO: note we may want to filter to have min reviews??

[(690952, 1), (213435, 2), (107828, 3), (66360, 4), (44928, 5), (32016, 6), (24555, 7), (19199, 8), (15451, 9), (12874, 10), (10695, 11), (9053, 12), (7843, 13), (6775, 14), (5974, 15), (5272, 16), (4702, 17), (4090, 18), (3774, 19), (3338, 20), (3078, 21), (2754, 22), (2649, 23), (2342, 24), (2202, 25), (1995, 26), (1879, 27), (1764, 28), (1615, 29), (1535, 30), (1464, 31), (1316, 32), (1259, 33), (1164, 34), (1110, 35), (1055, 36), (962, 37), (888, 38), (913, 39), (801, 40), (847, 41), (748, 42), (669, 43), (680, 44), (686, 45), (617, 46), (569, 47), (544, 48), (543, 49), (511, 50), (487, 51), (477, 52), (449, 53), (430, 54), (452, 55), (420, 56), (369, 57), (368, 58), (390, 59), (322, 60), (321, 61), (327, 62), (300, 63), (323, 64), (290, 65), (291, 66), (287, 67), (269, 68), (247, 69), (254, 70), (250, 71), (230, 72), (213, 73), (246, 74), (197, 75), (228, 76), (230, 77), (188, 78), (211, 79), (194, 80), (211, 81), (171, 82), (186, 83), (178, 84), (165, 85), (157, 86), (164, 87), (

In [ ]:
# find out how many samples we have given that we want n revies
def how_many_groups_of_n(n, freq_list):
    total = 0
    for freq, size in freq_list:
        if size >= n:
            total += freq

    return total

how_many_groups_of_n(14, freqs)


#95k books with at least 14 reviews, lets train with these


95499